## **One-hot encoding**

Generate embeddings for words based on all the words in the vocabulary

In [1]:
s1 = "The food is good"
s2 = "The food is bad"
s3 = "Pizza is amazing"

In [2]:
from nltk import word_tokenize

In [ ]:
sentence_set = []

for sentence in [s1, s2, s3]:
    sentence_set.extend(word_tokenize(sentence))

unique_words = list(set(sentence_set))

In [4]:
unique_words

['bad', 'is', 'amazing', 'Pizza', 'good', 'The', 'food']

In [5]:
default_vector = [0] * len(unique_words)

word_vector = {}

for idx, word in enumerate(unique_words):
    vector = default_vector.copy()
    vector[idx] = 1
    word_vector[word] = vector

### **Advantages**

* Easy to implement

### **Disadvantages**

* Generates sparse vectors which leads to overfitting
* Length of all vectors or inputs may not be the same. ML requires inputs to be of the same size
* No semantic meaning between the words is captured
* Vectors cannot be generated for words out of the vocabulary

In [6]:
word_vector

{'bad': [1, 0, 0, 0, 0, 0, 0],
 'is': [0, 1, 0, 0, 0, 0, 0],
 'amazing': [0, 0, 1, 0, 0, 0, 0],
 'Pizza': [0, 0, 0, 1, 0, 0, 0],
 'good': [0, 0, 0, 0, 1, 0, 0],
 'The': [0, 0, 0, 0, 0, 1, 0],
 'food': [0, 0, 0, 0, 0, 0, 1]}

In [ ]:
sent_vec = []

for sentence in [s1, s2, s3]:
    sentence_set = word_tokenize(sentence)
    sent_vec.append([word_vector[word] for word in sentence_set])

In [8]:
sent_vec

[[[0, 0, 0, 0, 0, 1, 0],
  [0, 0, 0, 0, 0, 0, 1],
  [0, 1, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 1, 0, 0]],
 [[0, 0, 0, 0, 0, 1, 0],
  [0, 0, 0, 0, 0, 0, 1],
  [0, 1, 0, 0, 0, 0, 0],
  [1, 0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 1, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0]]]

## **Bag of words**

* There are two ways to implement bag of words
    * For a repeated word, the vector value can only be set to 1
    * For a repeated word, the vector value can be incremented

In [9]:
s1 = "He is a good boy"
s2 = "She is a good girl"
s3 = "Boy and girl are good"

In [10]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

words_s1 = [word.lower() for word in word_tokenize(s1) if word.lower() not in stop_words]
words_s2 = [word.lower() for word in word_tokenize(s2) if word.lower() not in stop_words]
words_s3 = [word.lower() for word in word_tokenize(s3) if word.lower() not in stop_words]

### **Advantages**

* Easy to implement
* Each vector is of same size - Useful for ML

### **Disadvantages**

* Generates sparse vectors which leads to overfitting
* No semantic meaning between the words is captured
* Order of the words is lost
* Vectors cannot be generated for words out of the vocabulary

In [11]:
count = 0
bow = {}

for word in words_s1 + words_s2 + words_s3:
    if word not in bow:
        bow[word] = count
        count += 1

In [12]:
bow

{'good': 0, 'boy': 1, 'girl': 2}

In [13]:
default_vector = [0] * len(bow)
vector_s1 = default_vector.copy()
vector_s2 = default_vector.copy()
vector_s3 = default_vector.copy()

for word in words_s1:
    vector_s1[bow[word]] += 1

for word in words_s2:
    vector_s2[bow[word]] += 1

for word in words_s3:
    vector_s3[bow[word]] += 1

In [14]:
vector_s1, vector_s2, vector_s3

([1, 1, 0], [1, 0, 1], [1, 1, 1])

## **N-Grams**

To capture the semantic meaning of the sentences, we combine two or more features or unique words into one extra feature and check whether that is present in the provided sentence. This helps in capturing the semantic meaning of the sentences

In [15]:
def get_ngrams(words: list[str], n: int) -> list[str]:
    n_grams = []
    for i in range(len(words)-n+1):
        n_grams.append(" ".join(words[i:i+n]))
    
    return n_grams

In [16]:
words_s1.extend(get_ngrams(words_s1, 2))
words_s2.extend(get_ngrams(words_s2, 2))
words_s3.extend(get_ngrams(words_s3, 2))

In [17]:
words_s1, words_s2, words_s3

(['good', 'boy', 'good boy'],
 ['good', 'girl', 'good girl'],
 ['boy', 'girl', 'good', 'boy girl', 'girl good'])

In [18]:
count = len(bow)

new_bow = get_ngrams(list(bow.keys()), 2)

for word in new_bow:
    if word not in bow:
        bow[word] = count
        count += 1

In [19]:
bow

{'good': 0, 'boy': 1, 'girl': 2, 'good boy': 3, 'boy girl': 4}

In [21]:
default_vector_ngram = [0]*len(bow)

vector_s1_ngram = default_vector_ngram.copy()
vector_s2_ngram = default_vector_ngram.copy()
vector_s3_ngram = default_vector_ngram.copy()

for word in words_s1:
    if word in bow:
        vector_s1_ngram[bow[word]] += 1

for word in words_s2:
    if word in bow:
        vector_s2_ngram[bow[word]] += 1

for word in words_s3:
    if word in bow:
        vector_s3_ngram[bow[word]] += 1

In [22]:
vector_s1_ngram, vector_s2_ngram, vector_s3_ngram

([1, 1, 0, 1, 0], [1, 0, 1, 0, 0], [1, 1, 1, 0, 1])

## **TF-IDF -> Term Frequency - Inverse Document Frequency**

Generate vectors based on the term frequency and the inverse document frequency. Here, if a word is present in all the sentences, it is given less importance. Word importance is captured for every sentence in this technique.

In [30]:
from math import log

# Get the vocabulary from a list of sentences
def get_corpus(sentences: list[str]) -> list[str]:
    corpus = []
    
    for sentence in sentences:
        words = get_words(sentence)
        corpus.extend(words)
    
    return list(set(corpus))

# Get a list of words from a sentence
def get_words(text: str) -> list[str]:
    words = word_tokenize(text)
    words = [word.lower() for word in words if word.lower() not in stop_words]
    return words

# Get term frequency of a word from corpus in a sentence
# (no of repetitions of a word in a sentence/no of words in a sentence)
def get_tf(sentence: str, corpus: list[str]) -> dict[str, float]:
    tf = {}
    words = get_words(sentence)
    
    for word in words:
        if word not in tf:
            tf[word] = 0
        
        tf[word] += 1
    
    for word in corpus:
        tf[word] = tf.get(word, 0)/len(words)
    
    return tf

# Get inverse document frequency for each word in the corpus
# (no of sentences/no of sentences a word appears in)
def get_idf(corpus: list[str], sentences: list[str]) -> dict[str, float]:
    idf = {}
    n = len(sentences)
    sentences_set: list[set] = []

    for sentence in sentences:
        sentences_set.append(set(get_words(sentence)))

    for word in corpus:
        count = 0
        for sentence_set in sentences_set:
            if word in sentence_set:
                count += 1
        
        idf[word] = log(n/count)
    
    return idf

### **Advantages**

* Intuitive
* Fixed size vectors -> Useful for ML
* Word importance is getting captured

### **Disadvantages**

* Sparsity of vetors still exists
* Can't handle words out of the vocabulary

In [31]:
corpus = get_corpus([s1, s2, s3])

s1_tf = get_tf(s1, corpus)
s2_tf = get_tf(s2, corpus)
s3_tf = get_tf(s3, corpus)

idf = get_idf(corpus, [s1, s2, s3])

In [32]:
s1_tf, s2_tf, s3_tf, idf

({'good': 0.5, 'boy': 0.5, 'girl': 0.0},
 {'good': 0.5, 'girl': 0.5, 'boy': 0.0},
 {'boy': 0.3333333333333333,
  'girl': 0.3333333333333333,
  'good': 0.3333333333333333},
 {'good': 0.0, 'girl': 0.4054651081081644, 'boy': 0.4054651081081644})

In [33]:
s1_tf_idf = {}
s2_tf_idf = {}
s3_tf_idf = {}

for word in corpus:
    s1_tf_idf[word] = s1_tf[word] * idf[word]
    s2_tf_idf[word] = s2_tf[word] * idf[word]
    s3_tf_idf[word] = s3_tf[word] * idf[word]

In [34]:
s1_tf_idf, s2_tf_idf, s3_tf_idf

({'good': 0.0, 'girl': 0.0, 'boy': 0.2027325540540822},
 {'good': 0.0, 'girl': 0.2027325540540822, 'boy': 0.0},
 {'good': 0.0, 'girl': 0.13515503603605478, 'boy': 0.13515503603605478})